# FilPHANGS - Figure Production

Diagnostic and publication figures for the FilPHANGS pipeline:

1. **Pipeline walkthrough** -- six panels showing each processing stage applied to one galaxy
2. **Source removal validation** -- original vs. source-removed image with circled point sources
3. **Hierarchical RGB composite** -- filament composite maps at three scales as false-colour RGB

Edit `BASE_DIR`, `GALAXY`, and `BAND` in Cell 1 to switch to a different target.


In [ ]:
# =============================================================================
# Cell 1: Configuration and shared utilities
# Edit BASE_DIR, GALAXY, and BAND here; all subsequent cells use these variables.
# =============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle, Patch
from astropy.io import fits
from pathlib import Path
from skimage.measure import regionprops, label as sk_label
from skimage.draw import disk

# -- Edit these for your environment ------------------------------------------
BASE_DIR = Path(r"C:\Users\jhoffm72\Documents\FilPHANGS\Data")
GALAXY   = "ngc0628"
BAND     = "F770W"

galaxy_dir    = BASE_DIR / f"{GALAXY}_{BAND}"
orig_img_path = BASE_DIR / "OriginalImages" / f"{GALAXY}_{BAND}_JWST_Emission_starsub.fits"
FIGURES_DIR   = BASE_DIR / "Figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Binary colormap: dark purple = background, yellow = filaments
BINARY_CMAP = ListedColormap([(68/255, 1/255, 84/255), (1, 1, 0)])

def load_fits(path, hdu_idx=0):
    """Load a FITS HDU, replacing NaNs with 0."""
    with fits.open(path, ignore_missing=True) as h:
        return np.nan_to_num(np.array(h[hdu_idx].data, dtype=float))

def pct_clip(img, lo=2, hi=98):
    """Clip an image to the [lo, hi] percentile range."""
    return np.clip(img, np.percentile(img, lo), np.percentile(img, hi))


In [ ]:
# =============================================================================
# Cell 2: Pipeline Walkthrough Figure
# Six-panel figure showing each stage of the FilPHANGS pipeline applied to one
# galaxy at one CDD scale. Useful for methods sections in papers and talks.
# Update stage_files if your output naming differs from the FilPHANGS defaults.
# =============================================================================
SCALE   = 16         # CDD scale in parsecs
CY, CX  = 900, 700  # crop window top-left corner (row, col)
CROP_SZ = 400        # crop size in pixels

# Paths to each pipeline stage output
stage_files = [
    (galaxy_dir / "CDD" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_CDDss{SCALE:04d}pc.fits",
     f"1. {SCALE} pc CDD (arctan stretch)"),
    (galaxy_dir / "BkgSubDivRMS" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_CDDss{SCALE:04d}pc_BkgSubDivRMS.fits",
     "2. Background-subtracted / RMS"),
    (galaxy_dir / "SOAXOutput" / f"{SCALE}pc" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_CDDss{SCALE:04d}pc_Blocked--ridge0.02375--stretch1.750.fits",
     "3. Single SOAX run"),
    (galaxy_dir / "Composites" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_CDDss{SCALE:04d}pc_Composites.fits",
     "4. Stacked composite"),
    (galaxy_dir / "Composites" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_CDDss{SCALE:04d}pc_Composites.fits",
     "5. Skeletonized composite"),
    (galaxy_dir / "SyntheticMap" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_CDDss{SCALE:04d}pc_SyntheticMap_Grouped.fits",
     "6. PSF synthetic map"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)
for ax, (path, title) in zip(axes.flatten(), stage_files):
    try:
        img  = load_fits(path)
        crop = img[CY:CY+CROP_SZ, CX:CX+CROP_SZ]
        if len(np.unique(crop)) <= 2:    # binary skeleton -> use binary colourmap
            ax.imshow(crop, cmap=BINARY_CMAP, origin="lower", vmin=0, vmax=1)
        else:
            ax.imshow(pct_clip(crop), cmap="viridis", origin="lower")
    except FileNotFoundError:
        ax.text(0.5, 0.5, f"File not found:\n{Path(path).name}",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=7, color="red")
    ax.set_title(title, fontsize=11, weight="bold")
    ax.axis("off")

fig.suptitle(f"FilPHANGS Pipeline -- {GALAXY.upper()} {BAND} @ {SCALE} pc",
             fontsize=14, weight="bold")
out = FIGURES_DIR / f"Pipeline_{GALAXY}_{SCALE}pc.png"
fig.savefig(out, dpi=300)
plt.show()
print(f"Saved {out}")


In [ ]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from skimage.measure import regionprops, label
from matplotlib.patches import Circle
import os
from skimage.draw import disk

# File Paths
source_rem_path = r"C:\Users\jhoffm72\Documents\FilPHANGS\Data\ngc0628_F770W\Source_Removal\OriginalImageSourcesRemoved.fits"
orig_img_path = r"C:\Users\jhoffm72\Documents\FilPHANGS\Data\OriginalImages\ngc0628_F770W_JWST_Emission_starsub.fits"
mask_path = r"C:\Users\jhoffm72\Documents\FilPHANGS\Data\ngc0628_F770W\Source_Removal\_CDDfs0004pix_CDDfs0004pix_F770W_CDDfs_sources_S2N_mask.fits"

# Read source-removed image
with fits.open(source_rem_path) as hdu:
    source_removed_image = hdu[0].data[1]
    header_removed = hdu[0].header

# Read original image
with fits.open(orig_img_path) as hdu:
    orig_image = hdu[0].data
    header_orig = hdu[0].header
    orig_image[np.isnan(orig_image)] = 0

# Crop window
start_x, start_y = 700, 800
stop_x, stop_y = start_x + 300, start_y + 300

mask = fits.getdata(mask_path).astype(bool)

# Extract Circular Regions from Mask
labeled_mask = label(mask)
regions = regionprops(labeled_mask)

crop_origin = (start_y, start_x)
orig_crop = orig_image[start_y:stop_y, start_x:stop_x]
src_rem_crop = source_removed_image[start_y:stop_y, start_x:stop_x]

# Validate circles with combined rules
valid_regions = []
crop_height, crop_width = orig_crop.shape

bright_high = np.percentile(orig_crop, 95)  # automatically valid if >= this
bright_low = np.percentile(orig_crop, 5)    # automatically invalid if <= this

for region in regions:
    y, x = region.centroid
    radius = region.equivalent_diameter / 2

    if (start_y <= y < stop_y) and (start_x <= x < stop_x):
        y_rel = int(y - start_y)
        x_rel = int(x - start_x)
        rr, cc = disk((y_rel, x_rel), radius, shape=(crop_height, crop_width))

        region_orig = orig_crop[rr, cc]
        region_rem = src_rem_crop[rr, cc]

        mean_orig = np.mean(region_orig)
        mean_rem = np.mean(region_rem)
        diff_ratio = np.abs(mean_orig - mean_rem) / (mean_orig + 1e-8)

        # Skip if too dark
        if mean_orig <= bright_low:
            continue

        # Keep if difference >= 50% or very bright
        if diff_ratio >= 0.07 or mean_orig >= bright_high:
            valid_regions.append((x, y, radius))

print(f"{len(valid_regions)} valid circular regions (>=50% diff or bright, excluding dark regions).")

# Create Plot
fig, axs = plt.subplots(1, 2, figsize=(14, 7))

# Compute scaling from original crop
bright_percentile = 97
vmin, vmax = np.percentile(orig_crop, [1, bright_percentile])
bright_thresh = np.percentile(orig_crop, bright_percentile)
max_val = np.percentile(orig_crop, 99.8)  # or 99, depending on how much you want to clip


def show_image_with_circles_fixedscale(ax, image, crop_origin, vmin, vmax, bright_thresh, max_val):
    offset_y, offset_x = crop_origin
    ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)

    # Overlay for bright regions (using scaling from left panel)
    bright_mask = image >= bright_thresh
    bright_values = np.zeros_like(image)
    bright_values[bright_mask] = image[bright_mask]
    ax.imshow(bright_values, cmap='inferno', origin='lower',
              vmin=bright_thresh, vmax=max_val, alpha=(bright_values > 0) * 0.8)

    ax.axis('off')

    # Draw valid circles
    for x, y, radius in valid_regions:
        if (offset_y <= y < offset_y + image.shape[0]) and (offset_x <= x < offset_x + image.shape[1]):
            circ = Circle((x - offset_x, y - offset_y), radius,
                          edgecolor='red', facecolor='none', linewidth=1.5)
            ax.add_patch(circ)

# Use same scale for both panels
show_image_with_circles_fixedscale(axs[0], orig_crop, crop_origin, vmin, vmax, bright_thresh, max_val)
show_image_with_circles_fixedscale(axs[1], src_rem_crop, crop_origin, vmin, vmax, bright_thresh, max_val)

# Save Figure
out_dir = os.path.dirname(source_rem_path)
out_name = f"ValidatedCircles_{start_x}_{start_y}_Overlay.png"
out_path = os.path.join(out_dir, out_name)

plt.tight_layout()
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f" Figure saved to: {out_path}")

plt.show()


In [ ]:
# =============================================================================
# Cell 4: Multi-Scale Hierarchical RGB Composite
# Combines filament composite maps at three scales into a false-colour RGB image.
# Useful for visualising which spatial scales of structure are co-spatial.
#
# Colour key: Blue = 128 pc | Red = 64 pc | Green = 32 pc
# NOTE: uses Composite FITS files (stacked SOAX output), NOT synthetic maps.
#       For synthetic-map composites see SyntheticMap_Figure_Production.ipynb.
# =============================================================================
from scipy.ndimage import zoom

RGB_SCALES = {"blue": 128, "red": 64, "green": 32}   # scale (pc) -> channel colour

def load_composite(scale_pc):
    """Find and load the Composite FITS for this galaxy at the given scale."""
    comp_dir = galaxy_dir / "Composites"
    for fname in os.listdir(comp_dir):
        if fname.endswith(".fits") and f"{scale_pc:04d}pc" in fname:
            return np.nan_to_num(fits.getdata(comp_dir / fname).astype(float))
    raise FileNotFoundError(f"No composite for {scale_pc} pc in {comp_dir}")

def norm01(data, pct=99):
    """Normalise data to [0, 1], clipping at the given percentile."""
    lo, hi = np.min(data), np.percentile(data, pct)
    return np.clip((data - lo) / (hi - lo + 1e-9), 0, 1)

try:
    imgs   = {ch: load_composite(sc) for ch, sc in RGB_SCALES.items()}
    target = max(imgs.values(), key=lambda a: a.size).shape
    imgs   = {ch: zoom(a, (target[0]/a.shape[0], target[1]/a.shape[1]), order=1)
              for ch, a in imgs.items()}
    imgs   = {ch: norm01(a) for ch, a in imgs.items()}

    # Stack into RGB: channel order = (Red=64pc, Green=32pc, Blue=128pc)
    rgb = np.stack([imgs["red"], imgs["green"], imgs["blue"]], axis=-1)
    rgb = np.clip(rgb / (rgb.max() + 1e-9), 0, 1)

    legend = [Patch(color=ch, label=f"{sc} pc")
              for ch, sc in [("blue",128), ("red",64), ("green",32)]]

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(rgb, origin="lower")
    ax.axis("off")
    ax.set_title(f"{GALAXY.upper()} {BAND} -- Multi-Scale Filament Hierarchy", fontsize=12)
    ax.legend(handles=legend, loc="lower right", framealpha=0.7, fontsize=10)

    out = FIGURES_DIR / f"{GALAXY}_{BAND}_RGB_Composite.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")

except FileNotFoundError as e:
    print(f"Missing composite file: {e}")
    print("Run the FilPHANGS pipeline first to generate Composite FITS files.")
